# 🚀 LifeMate v3.0 — High-Speed Offline Inference Notebook
### Zalo AI Challenge 2026

This dedicated notebook runs **Inference Only** using the pre-trained weights from your Training Notebook:
1. **Input**: Connects to the training notebook output (`lifemate-qwen-sft` or `qwen-lora`).
2. **Audio ASR**: `openai/whisper-large-v3-turbo` (transcribes 634 audios, or reuses cached transcripts).
3. **LLM Agent**: Fine-tuned Qwen3-4B with loop-breaking, entity resolution, and fast early-stopping.
4. **Post-Processor**: Deduplication and `ToolEnv` state_hash validator ensuring 100% schema compliance.

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Install Core Dependencies
!pip install -q -U "transformers>=4.51.0" soundfile accelerate peft
!pip uninstall -y torchao

In [ ]:
# 3. Clone Repository with latest Agent & Cleaner fixes
import os
REPO_URL = "https://github.com/Quandinh030106/LifeMate-Voice2Action-Agent.git"
if not os.path.exists("/kaggle/working/LifeMate-Voice2Action-Agent"):
    !git clone $REPO_URL /kaggle/working/LifeMate-Voice2Action-Agent
else:
    !cd /kaggle/working/LifeMate-Voice2Action-Agent && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/LifeMate-Voice2Action-Agent

In [ ]:
# 4. Auto-Detect Dataset and Pre-Trained Weights from Input
import glob, os

# 1. Locate Data Directory
datasets = glob.glob("/kaggle/input/**/public_test.jsonl", recursive=True)
if datasets:
    DATA_DIR = os.path.dirname(os.path.dirname(datasets[0]))
    print("Found Data Directory:", DATA_DIR)
else:
    DATA_DIR = "/kaggle/input/lifemate-data/train+public_test/train+public_test"
    print("Defaulting to:", DATA_DIR)
os.environ["DATA_DIR"] = DATA_DIR

# 2. Locate Trained Model (either merged model directory or unmerged LoRA adapter)
candidate_models = glob.glob("/kaggle/input/**/lifemate-qwen-sft", recursive=True)
candidate_adapter_dirs = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)]
candidate_zips = glob.glob("/kaggle/input/**/qwen_lora*.zip", recursive=True)

if os.path.exists("/kaggle/working/lifemate-qwen-sft") and os.listdir("/kaggle/working/lifemate-qwen-sft"):
    MODEL_DIR = "/kaggle/working/lifemate-qwen-sft"
    print("Found local merged model at:", MODEL_DIR)
elif candidate_models:
    MODEL_DIR = candidate_models[0]
    print("Found Pre-Trained Merged Model in Input at:", MODEL_DIR)
elif candidate_adapter_dirs or candidate_zips:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch, gc, zipfile
    if candidate_zips and not candidate_adapter_dirs:
        unzip_dir = "/kaggle/working/qwen-lora"
        os.makedirs(unzip_dir, exist_ok=True)
        with zipfile.ZipFile(candidate_zips[0], 'r') as zf:
            zf.extractall(unzip_dir)
        adapter_dir = unzip_dir
    else:
        adapter_dir = candidate_adapter_dirs[0]
    print(f"Merging LoRA adapter from {adapter_dir} into base Qwen3-4B-Instruct-2507...")
    base_model_name = "Qwen/Qwen3-4B-Instruct-2507"
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map="cpu"
    )
    peft_model = PeftModel.from_pretrained(base_model, adapter_dir)
    merged_model = peft_model.merge_and_unload()
    MODEL_DIR = "/kaggle/working/lifemate-qwen-sft"
    os.makedirs(MODEL_DIR, exist_ok=True)
    merged_model.save_pretrained(MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    tokenizer.save_pretrained(MODEL_DIR)
    del base_model, peft_model, merged_model
    gc.collect()
    print("Merged Model saved to:", MODEL_DIR)
else:
    MODEL_DIR = "Qwen/Qwen3-4B-Instruct-2507"
    print("Notice: No custom weights found in Input. Defaulting to base:", MODEL_DIR)

os.environ["MODEL_DIR"] = MODEL_DIR

# Check for pre-existing audio transcripts to skip Whisper transcription completely
transcripts_cache = "/kaggle/working/transcripts_public_test.json"
found_transcripts = glob.glob("/kaggle/input/**/transcripts_public_test.json", recursive=True)
if found_transcripts and not os.path.exists(transcripts_cache):
    import shutil
    shutil.copy2(found_transcripts[0], transcripts_cache)
    print(f"Loaded cached transcripts from: {found_transcripts[0]}")

In [ ]:
# 5. Run Fast Inference on public_test (~15-20 mins total)
!python -u inference.py \
    --data_dir "$DATA_DIR" \
    --split public_test \
    --asr_engine openai \
    --asr_model openai/whisper-large-v3-turbo \
    --llm_model "$MODEL_DIR" \
    --output /kaggle/working/submission.jsonl

In [ ]:
# 6. Clean, Deduplicate, and Auto-Repair Submission
!python scripts/clean_submission.py --input /kaggle/working/submission.jsonl --data_dir "$DATA_DIR"

In [ ]:
# 7. Validate with Official zaic_tools Validator (Must pass with 0 errors)
import os, subprocess
env = os.environ.copy()
env["PYTHONPATH"] = f"{DATA_DIR}/public_test:" + env.get("PYTHONPATH", "")
cmd = [
    "python", "-m", "zaic_tools",
    "--split", "public_test",
    "--root", DATA_DIR,
    "--validate", "/kaggle/working/submission.jsonl"
]
print("Running official zaic_tools validator...")
res = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)

In [ ]:
# 8. Verify Line Count (Must be exactly 634 lines)
!wc -l /kaggle/working/submission.jsonl
!head -n 5 /kaggle/working/submission.jsonl